In [ ]:
// load the formatting setup
#load "../setup.fsx"

# The Set $\mathbb{Q}$ is Countable

Although `F#` has no built-in type for rational numbers, we can use the type `BigRational` from the library
[MathNet.Numerics](https://numerics.mathdotnet.com).  This library is downloaded from the package manager *NuGet* the first time the next cell is executed.
A value of type `BigRational` is a fraction $\frac{p}{q}$ where both $p$ and $q$ are arbitrary large integers.  Fractions are always stored in reduced form.

Rational literals are written with the suffix `N`, e.g. `1N/2N` denotes the fraction $\frac{1}{2}$.

In [ ]:
#r "nuget: MathNet.Numerics.FSharp"
open MathNet.Numerics

In [ ]:
1N/2N + 1N/3N

The function `gcd` takes two arguments:
- `a` is a natural number,
- `b` is a natural number.

The function returns the *greatest common divisor* of `a` and `b`.

**Implementation:** The function implements the *Euclidean algorithm*, which is based on the following two equations:
- $\texttt{gcd}(a, 0) = a$,
- $\texttt{gcd}(a, b) = \texttt{gcd}(b, a \,\texttt{%}\, b)$ if $b \not= 0$.

In [ ]:
let rec gcd (a: int) (b: int) : int =
    if b = 0 then
        a
    else
        gcd b (a % b)

In [ ]:
gcd 120 100

The value `rationals` is an infinite *sequence* that enumerates all rational numbers.  In `F#`, a sequence of type `seq<'T>` is *lazy*: its elements are only computed when they are needed.  Therefore, a sequence can be infinite.  Sequences are defined using *sequence expressions* of the form `seq { ... }`, where every `yield` produces the next element of the sequence.  Sequence expressions play the same role as *generators* in Python.

The rational numbers are enumerated as follows:
* First, the rational number $\frac{0}{1}$ is generated.
* Second, all rational numbers $\frac{p}{q}$ and $-\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 2$ and $\texttt{gcd}(p, q) = 1$,
* Third, all rational numbers $\frac{p}{q}$ and $-\frac{p}{q}$ are computed that satisfy $p > 0$, $p + q = 3$ and $\texttt{gcd}(p, q) = 1$,
* $\vdots$

**Implementation:** `Seq.initInfinite (fun i -> i + 2)` is the infinite sequence $2, 3, 4, \cdots$.  The variable `sum` runs through this sequence.  For every value of `sum` we generate all fractions $\frac{p}{q}$ such that $p + q = \texttt{sum}$, $p > 0$, and $p$ and $q$ are coprime.  The condition $\texttt{gcd}(p, q) = 1$ ensures that every rational number is enumerated exactly once.  The function `BigRational.FromIntFraction(p, q)` creates the fraction $\frac{p}{q}$.

In [ ]:
let rationals : seq<BigRational> =
    seq {
        yield 0N  
        for sum in Seq.initInfinite (fun i -> i + 2) do
            for numerator in 1 .. sum - 1 do
                let denominator = sum - numerator
                if gcd numerator denominator = 1 then
                    yield  BigRational.FromIntFraction(numerator, denominator)
                    yield -BigRational.FromIntFraction(numerator, denominator)
    }

The value `generateRationals` is an infinite sequence of pairs of the form
`(n, q)` where `n` is a positive natural number and `q` is the $n^\mathrm{th}$ rational number of the enumeration given by `rationals`.

**Implementation:** The function `Seq.mapi` applies a function to every element of a sequence.  In contrast to `Seq.map`, the function also receives the index `i` of the element, where the index of the first element is $0$.  Hence the $n^\mathrm{th}$ element has the index $n-1$.

In [ ]:
let generateRationals : seq<int * BigRational> =
    Seq.mapi (fun i q -> (i + 1, q)) rationals

The function `gen` takes one argument:
- `cnt` is a natural number.

The function prints the first `cnt` pairs of the sequence `generateRationals`, one pair per line.  It does not return a meaningful result.

**Implementation:** `Seq.take cnt` takes the first `cnt` elements of the infinite sequence and `Seq.iter` prints each of them.
The format `"%A"` prints a value using F# syntax.  Note that `"%A"` prints a `BigRational` using the suffix `N`, e.g. the fraction $\frac{1}{2}$ is printed as `1/2N`.

In [ ]:
let gen (cnt: int) : unit =
    generateRationals
    |> Seq.take cnt
    |> Seq.iter (printfn "%A")

In [ ]:
gen 1000